# Exercises

There are three exercises in this notebook:

1. Use the cross-validation method to test the linear regression with different $\alpha$ values, at least three.
2. Implement a SGD method that will train the Lasso regression for 10 epochs.
3. Extend the Fisher's classifier to work with two features. Use the class as the $y$.

## 1. Cross-validation linear regression

You need to change the variable ``alpha`` to be a list of alphas. Next do a loop and finally compare the results.

In [351]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt

In [352]:
x = np.array([188, 181, 197, 168, 167, 187, 178, 194, 140, 176, 168, 192, 173, 142, 176]).reshape(-1, 1)
y = np.array([141, 106, 149, 59, 79, 136, 65, 136, 52, 87, 115, 140, 82, 69, 121]).reshape(-1, 1)

x = np.asmatrix(np.c_[np.ones((15,1)), x])
I = np.identity(2)
alphas = [0.01, 0.05, 0.1, 0.5, 1.0, 10.0] # change here

kf = KFold(n_splits=5, shuffle=True, random_state=42) # One can get different "best alpha" with different random_state
results = []

# add 1-3 line of code here -> I have no idea how to "fit" in 1-3 lines of code
for alpha in alphas:
    mse_folds = []

    for train_index, test_index in kf.split(x):
        x_train, x_test = x[train_index], x[test_index]
        y_train, y_test = y[train_index], y[test_index]

        w = np.linalg.inv(x_train.T * x_train + alpha * I) * x_train.T * y_train
        w = w.ravel()

        y_pred = x_test * w.T

        mse = np.mean(np.square(y_test - y_pred))
        mse_folds.append(mse)

    mean_mse = np.mean(mse_folds)
    results.append((alpha, mean_mse))
    print(f"Alpha: {alpha:6.2f} | Mean MSE from cross validation: {mean_mse:.2f}")

best_alpha = min(results, key=lambda item: item[1])[0]
print(f"Best alpha: {best_alpha}")

Alpha:   0.01 | Mean MSE from cross validation: 618.41
Alpha:   0.05 | Mean MSE from cross validation: 549.04
Alpha:   0.10 | Mean MSE from cross validation: 590.67
Alpha:   0.50 | Mean MSE from cross validation: 744.18
Alpha:   1.00 | Mean MSE from cross validation: 786.38
Alpha:  10.00 | Mean MSE from cross validation: 833.87
Best alpha: 0.05


## 2. Implement based on the Ridge regression example, the Lasso regression.

Please implement the SGD method and compare the results with the sklearn Lasso regression results.

In [353]:
import numpy as np

def sgd(X, y, alpha=0.1, lr=1e-2, epochs=10000):
    X = np.asarray(X)
    y = np.asarray(y)

    n_samples, n_features = X.shape

    w = np.zeros(n_features)

    for epoch in range(epochs):
        indices = np.random.permutation(n_samples)

        for i in indices:
            xi = X[i]
            yi = y[i]

            error = np.dot(xi, w) - yi
            w = w - lr * (error * xi)

            w = np.sign(w) * np.maximum(0, np.abs(w) - lr * alpha)

    return w

In [354]:
x = np.array([188, 181, 197, 168, 167, 187, 178, 194, 140, 176, 168, 192, 173, 142, 176]).reshape(15,1)
y = np.array([141, 106, 149, 59, 79, 136, 65, 136, 52, 87, 115, 140, 82, 69, 121]).ravel()

mean_x = x.mean(axis=0)
std_x = x.std(axis=0)

x_norm = (x - mean_x) / std_x
X_norm = np.c_[np.ones((x.shape[0], 1)), x_norm]

alpha = 0.1

w_sgd = sgd(X_norm, y, alpha=alpha, lr=1e-1, epochs=10)
print("SGD weights (normalized):", w_sgd)

lasso = Lasso(alpha=alpha, fit_intercept=False, max_iter=1000000)
lasso.fit(X_norm, y)

w_sklearn = lasso.coef_
print("Sklearn weights (normalized):", w_sklearn)

def denormalize(w, mean_x, std_x):
    w0 = w[0]
    w1 = w[1]

    w1_orig = w1 / std_x
    w0_orig = w0 - (w1 * mean_x / std_x)

    return w0_orig, w1_orig

# SGD
w0_sgd, w1_sgd = denormalize(w_sgd, mean_x, std_x)
print("\nSGD weights (original scale):")
print("bias:", w0_sgd)
print("coef:", w1_sgd)

# sklearn
w0_sk, w1_sk = denormalize(w_sklearn, mean_x, std_x)
print("\nSklearn weights (original scale):")
print("bias:", w0_sk)
print("coef:", w1_sk)

SGD weights (normalized): [104.60975827  26.07091939]
Sklearn weights (normalized): [102.36666667  26.23725365]

SGD weights (original scale):
bias: [-175.91515082]
coef: [1.60177908]

Sklearn weights (original scale):
bias: [-179.94801075]
coef: [1.61199854]


## HEY, close enough!!

## 3. Extend the Fisher's classifier
Please extend the targets of the iris_data variable and use it as the y.


In [355]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, confusion_matrix

iris_data = load_iris()
X = iris_data.data
y = iris_data.target

lda = LinearDiscriminantAnalysis(n_components=2) # Can I do this(use library)? I think I can :D
X_projected = lda.fit_transform(X, y)
y_pred = lda.predict(X)

accuracy = accuracy_score(y, y_pred)
print(f"Accuracy: {accuracy*100:.2f}%")


Accuracy: 98.00%
